In [23]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report


In [24]:
dataset1 = pd.read_csv("prep.csv")
df2=dataset1
df2 = pd.get_dummies(df2, drop_first=True)
indep_x = df2.drop('classification_yes', axis=1)
dep_y = df2['classification_yes']

In [25]:
#def selectKBest(indep_x, dep_y, n):
    #test = SelectKBest(chi2, k=n)
    #fit1 = test.fit(indep_x, dep_y)
    #selectk_features = fit1.transform(indep_x)
    #selected_columns = indep_x.columns[fit1.get_support()]
    #return selectk_features, selected_columns

#Apply PCA


def split_scalar(indep_x, dep_y):
    x_train, x_test, y_train, y_test = train_test_split(indep_x, dep_y, test_size = 0.25, random_state=0)
    sc = StandardScaler()
    x_train = sc.fit_transform(x_train)
    x_test = sc.transform(x_test)
    from sklearn.decomposition import PCA
    pca = PCA(n_components = 2)
    x_train = pca.fit_transform(x_train)
    x_test = pca.transform(x_test)
    explained_variance = pca.explained_variance_ratio_
    return x_train, x_test, y_train, y_test
    

In [26]:
#Kbest,ColBest = selectKBest(indep_x, dep_y, 3)
x_train, x_test, y_train, y_test = split_scalar(indep_x, dep_y)


In [27]:
def cm_prediction(Classifier, x_test, y_test):
    from sklearn.metrics import confusion_matrix
    from sklearn.metrics import accuracy_score
    from sklearn.metrics import classification_report
    y_pred = Classifier.predict(x_test)
    cm = confusion_matrix(y_test, y_pred)
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return Classifier, Accuracy, report, x_test, y_test, cm


In [28]:
#Logistic
def logistic(x_train, y_train, x_test, y_test):
    from sklearn.linear_model import LogisticRegression

    Classifier = LogisticRegression(random_state=0)

    Classifier.fit(x_train, y_train)

    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(
        Classifier, x_test, y_test
    )

    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMLinear
def svm_linear(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'linear', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#SVMNonLinear
def svm_NL(x_train, y_train, x_test, y_test):
    from sklearn.svm import SVC
    Classifier = SVC(kernel = 'rbf', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#Navie
def Navie(x_train, y_train, x_test, y_test):
    from sklearn.naive_bayes import GaussianNB
    Classifier = GaussianNB()
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#KNN
def knn(x_train, y_train, x_test, y_test):
    from sklearn.neighbors import KNeighborsClassifier
    Classifier = KNeighborsClassifier(n_neighbors = 5, metric = 'minkowski', p=2)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm
#Decision
def Decision(x_train, y_train, x_test, y_test):
    from sklearn.tree import DecisionTreeClassifier
    Classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier, x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

#RandomForest
def random(x_train, y_train, x_test, y_test):
    from sklearn.ensemble import RandomForestClassifier
    Classifier = RandomForestClassifier(n_estimators =10, criterion = 'entropy', random_state=0)
    Classifier.fit(x_train, y_train)
    Classifier, Accuracy, report, x_test, y_test, cm = cm_prediction(Classifier,x_test,y_test)
    return Classifier, Accuracy, report, x_test, y_test, cm

In [29]:
def PCA_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf):
    dataframe = pd.DataFrame(index=['pca=2'], columns=['Logistic', 'SVML', 'SVMNL', 'KNN', 'Navie', 'Decision', 'Random'])

    for number, idex in enumerate(dataframe.index):
        dataframe.loc[idex, 'Logistic']=acclog[number]
        dataframe.loc[idex, 'SVML'] = accsvml[number]
        dataframe.loc[idex,'SVMNL'] = accsvmnl[number]
        dataframe.loc[idex, 'KNN'] = accknn[number]
        dataframe.loc[idex, 'Navie'] = accnav[number]
        dataframe.loc[idex, 'Decision'] = accdes[number]
        dataframe.loc[idex, 'Random'] = accrf[number]
        return dataframe

In [30]:
acclog = []
accsvml = []
accsvmnl = []
accknn = []
accnav = []
accdes = []
accrf = []

Classifier, Accuracy, report, x_test, y_test, cm = logistic(x_train, y_train, x_test, y_test)
acclog.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = svm_linear(x_train, y_train, x_test, y_test)
accsvml.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = svm_NL(x_train, y_train, x_test, y_test)
accsvmnl.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = knn(x_train, y_train, x_test, y_test)
accknn.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = Navie(x_train, y_train, x_test, y_test)
accnav.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = Decision(x_train, y_train, x_test, y_test)
accdes.append(Accuracy)

Classifier, Accuracy, report, x_test, y_test, cm = random(x_train, y_train, x_test, y_test)
accrf.append(Accuracy)



In [31]:
result_pca_df = PCA_Classification(acclog, accsvml, accsvmnl, accknn,accnav, accdes, accrf)

# Comparison of each Model's Accuracy value

In [32]:
result_pca_df

,Logistic,SVML,SVMNL,KNN,Navie,Decision,Random
pca=2,0.99,0.99,1.0,1.0,1.0,0.99,0.99


In [33]:
print(acclog)
print(accsvml)
print(accsvmnl)
print(accknn)
print(accnav)
print(accdes)
print(accrf)

[0.99]
[0.99]
[1.0]
[1.0]
[1.0]
[0.99]
[0.99]
